In [ ]:
import sys
sys.path.insert(0, '..')

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
import optax

from src.data_gen import (
    generate_training_dataset,
    generate_slowness_map_2d,
    sample_collocation_points_2d
)
from src.model import WavePINN, WavePINNConfig, predict_on_grid
from src.inverse import (
    InverseProblem,
    InverseConfig,
    generate_synthetic_observations,
    evaluate_slowness_reconstruction
)

print(f"JAX devices: {jax.devices()}")

## 1. Generate Ground Truth Model

In [ ]:
# Generate a simple ground-truth velocity model
# Using fewer anomalies for easier reconstruction
print("Generating ground truth velocity model...")

nx, nz = 50, 50
true_slowness, true_velocity, spatial_coords = generate_slowness_map_2d(
    nx=nx, nz=nz,
    base_velocity=2000.0,
    n_anomalies=2,
    anomaly_strength=0.2,
    seed=42
)

print(f"Velocity range: [{float(true_velocity.min()):.0f}, {float(true_velocity.max()):.0f}] m/s")

# Visualize ground truth
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(true_velocity.T, origin='lower', cmap='viridis', extent=[0, 1, 0, 1])
axes[0].set_title('True Velocity (m/s)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('z')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(true_slowness.T * 1000, origin='lower', cmap='plasma', extent=[0, 1, 0, 1])
axes[1].set_title('True Slowness (ms/m)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('z')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## 2. Train Forward Model (to Generate Observations)

In [ ]:
# First, train a forward model with the TRUE velocity to generate observations
print("Training forward model with true velocity...")

# Generate training data with true velocity
data = generate_training_dataset(
    seed=42,
    nx=nx, nz=nz,
    n_interior=3000,
    n_boundary=500,
    n_initial=300,
    base_velocity=2000.0,
    n_anomalies=2
)

# Use our generated velocity model
from src.data_gen import interpolate_velocity_at_coords
velocity_at_interior = interpolate_velocity_at_coords(
    true_velocity, 
    data['collocation']['interior'][:, :2]
)

# Initialize forward model
forward_config = WavePINNConfig(
    hidden_dims=[64, 64, 32],
    use_fourier_features=True,
    num_fourier_features=32
)
forward_pinn = WavePINN(forward_config, seed=42)
key = jr.PRNGKey(42)
forward_params = forward_pinn.init_params(key)

# Training batch with true velocity
forward_batch = {
    'interior': data['collocation']['interior'],
    'boundary': data['collocation']['boundary'],
    'initial': data['collocation']['initial'],
    'velocity': velocity_at_interior
}

# Train forward model
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(forward_params)

@jax.jit
def train_step_forward(params, opt_state, batch):
    loss, grads = jax.value_and_grad(lambda p: forward_pinn.total_loss(p, batch))(params)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss

print("Training forward model (100 epochs)...")
for epoch in range(100):
    forward_params, opt_state, loss = train_step_forward(forward_params, opt_state, forward_batch)
    if epoch % 20 == 0:
        print(f"  Epoch {epoch}: Loss = {loss:.4e}")

print(f"Forward model trained! Final loss: {loss:.4e}")

## 3. Generate Synthetic Observations

In [ ]:
# Generate observed data at boundary points
# In practice, these would come from real measurements
print("Generating synthetic boundary observations...")

key = jr.PRNGKey(123)
observed_data = generate_synthetic_observations(
    pinn=forward_pinn,
    params=forward_params,
    boundary_coords=data['collocation']['boundary'],
    noise_level=0.01,  # 1% noise
    key=key
)

print(f"Observation points: {observed_data['coords'].shape[0]}")
print(f"Observation range: [{float(observed_data['values'].min()):.4f}, {float(observed_data['values'].max()):.4f}]")

# Visualize observations
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    observed_data['coords'][:, 0],
    observed_data['coords'][:, 2],  # time
    c=observed_data['values'],
    cmap='seismic',
    s=10,
    alpha=0.7
)
ax.set_xlabel('Position (x or z on boundary)')
ax.set_ylabel('Time (s)')
ax.set_title('Observed Boundary Traces')
plt.colorbar(sc, label='Wavefield u')
plt.tight_layout()
plt.show()

## 4. Setup and Run Inverse Problem

In [ ]:
# Configure inverse problem
print("Setting up inverse problem...")

inverse_config = InverseConfig(
    slowness_hidden_dims=[64, 64, 32],
    use_fourier_slowness=True,
    lambda_data=100.0,
    lambda_pde=1.0,
    lambda_bc=10.0,
    lambda_ic=10.0,
    lambda_smooth=0.1,
    min_velocity=1000.0,
    max_velocity=4000.0,
    learning_rate=1e-3,
    n_iterations=500,
    log_every=50
)

# Create wavefield model for inverse (starts fresh)
wave_config = WavePINNConfig(
    hidden_dims=[64, 64, 32],
    use_fourier_features=True,
    num_fourier_features=32
)
wave_pinn = WavePINN(wave_config, seed=123)

# Create inverse problem solver
inverse_solver = InverseProblem(
    wavefield_pinn=wave_pinn,
    config=inverse_config,
    seed=123
)

print("Inverse problem configured.")

In [ ]:
# Run inverse optimization
print("\nRunning inverse optimization...")
print("(This jointly optimizes wavefield u_θ and slowness m_φ)\n")

# Prepare batch for inverse
inverse_batch = {
    'interior': data['collocation']['interior'],
    'boundary': data['collocation']['boundary'],
    'initial': data['collocation']['initial']
}

key = jr.PRNGKey(456)
wave_params_inv, slowness_params, inv_history = inverse_solver.invert(
    batch=inverse_batch,
    observed_data=observed_data,
    key=key
)

## 5. Evaluate Reconstruction

In [ ]:
# Predict slowness on grid
print("Evaluating reconstruction...")

# Create evaluation grid
x_eval = jnp.linspace(0, 1, nx)
z_eval = jnp.linspace(0, 1, nz)
xx, zz = jnp.meshgrid(x_eval, z_eval, indexing='ij')
eval_coords = jnp.stack([xx.ravel(), zz.ravel()], axis=-1)

# Predict slowness
pred_slowness_flat = inverse_solver.predict_slowness(slowness_params, eval_coords)
pred_slowness = pred_slowness_flat.reshape(nx, nz)
pred_velocity = 1.0 / pred_slowness

# Compute metrics
metrics = evaluate_slowness_reconstruction(
    true_slowness, pred_slowness, eval_coords
)

print(f"\nReconstruction Metrics:")
print(f"  Slowness MSE: {metrics['slowness_mse']:.4e}")
print(f"  Slowness MAE: {metrics['slowness_mae']:.4e}")
print(f"  Relative Error: {metrics['relative_error']*100:.2f}%")
print(f"  Velocity MSE: {metrics['velocity_mse']:.4e}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Velocity comparison
vmin_v = min(float(true_velocity.min()), float(pred_velocity.min()))
vmax_v = max(float(true_velocity.max()), float(pred_velocity.max()))

im00 = axes[0, 0].imshow(true_velocity.T, origin='lower', cmap='viridis', 
                         extent=[0, 1, 0, 1], vmin=vmin_v, vmax=vmax_v)
axes[0, 0].set_title('True Velocity (m/s)')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('z')
plt.colorbar(im00, ax=axes[0, 0])

im01 = axes[0, 1].imshow(pred_velocity.T, origin='lower', cmap='viridis',
                         extent=[0, 1, 0, 1], vmin=vmin_v, vmax=vmax_v)
axes[0, 1].set_title('Reconstructed Velocity (m/s)')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('z')
plt.colorbar(im01, ax=axes[0, 1])

# Velocity error
vel_error = jnp.abs(true_velocity - pred_velocity)
im02 = axes[0, 2].imshow(vel_error.T, origin='lower', cmap='hot',
                         extent=[0, 1, 0, 1])
axes[0, 2].set_title('Velocity Error |True - Pred| (m/s)')
axes[0, 2].set_xlabel('x')
axes[0, 2].set_ylabel('z')
plt.colorbar(im02, ax=axes[0, 2])

# Slowness comparison
vmin_s = min(float(true_slowness.min()), float(pred_slowness.min())) * 1000
vmax_s = max(float(true_slowness.max()), float(pred_slowness.max())) * 1000

im10 = axes[1, 0].imshow(true_slowness.T * 1000, origin='lower', cmap='plasma',
                         extent=[0, 1, 0, 1], vmin=vmin_s, vmax=vmax_s)
axes[1, 0].set_title('True Slowness (ms/m)')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('z')
plt.colorbar(im10, ax=axes[1, 0])

im11 = axes[1, 1].imshow(pred_slowness.T * 1000, origin='lower', cmap='plasma',
                         extent=[0, 1, 0, 1], vmin=vmin_s, vmax=vmax_s)
axes[1, 1].set_title('Reconstructed Slowness (ms/m)')
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('z')
plt.colorbar(im11, ax=axes[1, 1])

# Slowness error
slow_error = jnp.abs(true_slowness - pred_slowness) * 1000
im12 = axes[1, 2].imshow(slow_error.T, origin='lower', cmap='hot',
                         extent=[0, 1, 0, 1])
axes[1, 2].set_title('Slowness Error (ms/m)')
axes[1, 2].set_xlabel('x')
axes[1, 2].set_ylabel('z')
plt.colorbar(im12, ax=axes[1, 2])

plt.suptitle('Inverse Problem: Slowness/Velocity Reconstruction', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Inverse Problem Loss Curves

In [ ]:
# Plot inverse problem loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps = [h['step'] for h in inv_history]
total_loss = [h['total'] for h in inv_history]
data_loss = [h['data'] for h in inv_history]
pde_loss = [h['pde'] for h in inv_history]

# Total loss
axes[0].semilogy(steps, total_loss, 'b-o', linewidth=2, label='Total')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Inverse Problem: Total Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Component losses
axes[1].semilogy(steps, data_loss, 'r-o', linewidth=2, label='Data Misfit')
axes[1].semilogy(steps, pde_loss, 'g-s', linewidth=2, label='PDE Residual')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss Component')
axes[1].set_title('Loss Components')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Cross-section Comparison

In [ ]:
# Compare velocity profiles along cross-sections
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Horizontal cross-section at z=0.5
z_idx = nz // 2
x_axis = jnp.linspace(0, 1, nx)

axes[0].plot(x_axis, true_velocity[:, z_idx], 'b-', linewidth=2, label='True')
axes[0].plot(x_axis, pred_velocity[:, z_idx], 'r--', linewidth=2, label='Reconstructed')
axes[0].set_xlabel('x')
axes[0].set_ylabel('Velocity (m/s)')
axes[0].set_title(f'Horizontal Profile at z = 0.5')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Vertical cross-section at x=0.5
x_idx = nx // 2
z_axis = jnp.linspace(0, 1, nz)

axes[1].plot(true_velocity[x_idx, :], z_axis, 'b-', linewidth=2, label='True')
axes[1].plot(pred_velocity[x_idx, :], z_axis, 'r--', linewidth=2, label='Reconstructed')
axes[1].set_xlabel('Velocity (m/s)')
axes[1].set_ylabel('z')
axes[1].set_title(f'Vertical Profile at x = 0.5')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 60)
print("INVERSE PROBLEM SUMMARY")
print("=" * 60)

print(f"\nTrue Velocity Model:")
print(f"  Range: [{float(true_velocity.min()):.0f}, {float(true_velocity.max()):.0f}] m/s")
print(f"  Mean: {float(true_velocity.mean()):.0f} m/s")

print(f"\nReconstructed Velocity Model:")
print(f"  Range: [{float(pred_velocity.min()):.0f}, {float(pred_velocity.max()):.0f}] m/s")
print(f"  Mean: {float(pred_velocity.mean()):.0f} m/s")

print(f"\nReconstruction Quality:")
print(f"  Relative Error: {metrics['relative_error']*100:.2f}%")
print(f"  Velocity RMSE: {np.sqrt(metrics['velocity_mse']):.1f} m/s")

print(f"\nInverse Optimization:")
print(f"  Initial loss: {inv_history[0]['total']:.4e}")
print(f"  Final loss: {inv_history[-1]['total']:.4e}")
print(f"  Final data misfit: {inv_history[-1]['data']:.4e}")

print("\n" + "=" * 60)